In [1]:
import torch

#Mask

In [10]:
seq_len=4
torch.ones(seq_len,seq_len)

tensor([[1., 1., 1., 1.],
        [1., 1., 1., 1.],
        [1., 1., 1., 1.],
        [1., 1., 1., 1.]])

In [6]:
seq_len=4
mask=torch.tril(
    torch.ones(seq_len,seq_len)
)
print(mask)

tensor([[1., 0., 0., 0.],
        [1., 1., 0., 0.],
        [1., 1., 1., 0.],
        [1., 1., 1., 1.]])


#BoolenMask

In [8]:
mask=torch.tril(
    torch.ones(seq_len,seq_len)
).bool()
mask

tensor([[ True, False, False, False],
        [ True,  True, False, False],
        [ True,  True,  True, False],
        [ True,  True,  True,  True]])

In [9]:
scores=torch.randn(4,4)
scores

tensor([[-0.4178, -0.1585,  1.1416, -0.4485],
        [-1.3878,  0.7011, -0.1307,  0.8625],
        [-0.5033, -0.1026, -0.5387,  0.5674],
        [ 1.9193, -0.2421, -0.6140,  1.0403]])

#apply mask

In [11]:
scores=scores.masked_fill(
    mask==0,
    float("-inf")
)
scores

tensor([[-0.4178,    -inf,    -inf,    -inf],
        [-1.3878,  0.7011,    -inf,    -inf],
        [-0.5033, -0.1026, -0.5387,    -inf],
        [ 1.9193, -0.2421, -0.6140,  1.0403]])

#Softmax

In [13]:
weights=torch.softmax(
    scores,dim=-1
)
weights

tensor([[1.0000, 0.0000, 0.0000, 0.0000],
        [0.1102, 0.8898, 0.0000, 0.0000],
        [0.2892, 0.4317, 0.2791, 0.0000],
        [0.6212, 0.0715, 0.0493, 0.2579]])

#Masked Multi-Head Attention

## Full Architecture



```
Input
   │
   ▼
Linear(Q)
Linear(K)
Linear(V)
   │
   ▼
Split Heads
   │
   ▼
QKᵀ
   │
   ▼
Scale
   │
   ▼
Apply Causal Mask
   │
   ▼
Softmax
   │
   ▼
Weights × V
   │
   ▼
Merge Heads
   │
   ▼
Final Linear
```



In [14]:
import math
import torch
import torch.nn as nn

In [35]:
class MaskedMultiHeadAttention(nn.Module):
  def __init__(self,d_model,num_heads):
    super().__init__()

    assert d_model%num_heads==0
    self.d_model=d_model
    self.num_heads=num_heads
    self.head_dim=d_model//num_heads

    self.Wq=nn.Linear(d_model,d_model)
    self.Wk=nn.Linear(d_model,d_model)
    self.Wv=nn.Linear(d_model,d_model)

    self.fc=nn.Linear(d_model,d_model)

  def forward(self,x):
    batch_size=x.size(0)
    seq_len=x.size(1)

    Q=self.Wq(x)
    K=self.Wk(x)
    V=self.Wv(x)

    # split into heads
    Q=Q.view(
        batch_size,seq_len,
        self.num_heads,self.head_dim
    ).transpose(1,2)
    K=K.view(
        batch_size,seq_len,
        self.num_heads,self.head_dim
    ).transpose(1,2)
    V=V.view(
        batch_size,seq_len,
        self.num_heads,self.head_dim
    ).transpose(1,2)

    #scores
    scores=torch.matmul(
        Q,
        K.transpose(-2,-1)
    )
    #scaling
    scores=scores/math.sqrt(
        self.head_dim
    )

    # mask
    mask=torch.tril(
        torch.ones(
            seq_len,seq_len,
            device=x.device
        )
    ).bool()

    scores=scores.masked_fill(
        mask==0,
        float("-inf")
    )
    #attention weight(softmax)
    weights=torch.softmax(
        scores,dim=-1
    )

    # context
    output=torch.matmul(
        weights,V
    )

    output=output.transpose(
        1,2
    ).contiguous().view(
        batch_size,seq_len,self.d_model
    )

    output=self.fc(output)
    return output


In [36]:
d_model=8
num_heads=2
batch_size=2
seq_len =5
attention=MaskedMultiHeadAttention(
    d_model,num_heads
)
x=torch.randn(
    batch_size,
    seq_len,
    d_model
)
print("Input Shape: ",x.shape)
output=attention(x)
print("="*50)
print("Output Shape: ",output.shape)

Input Shape:  torch.Size([2, 5, 8])
Output Shape:  torch.Size([2, 5, 8])
